In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pwd
!mkdir ./medgemma_output
!ls /kaggle/working/

# MedGemma 1.5 ファインチューニング
## ターミナルケア看護師向けメンタルヘルスカウンセリング

このノートブックでは、Google MedGemma 1.5 (4B) を
ターミナルケア看護師のメンタルヘルス維持のための専門家カウンセリングにファインチューニングします。

### 主な特徴
- QLoRAを使用したメモリ効率の良いファインチューニング
- 10,000件の専門家カウンセリング会話データを使用

In [ ]:
# 必要なライブラリのインストール
!pip install -q transformers accelerate bitsandbytes peft trl datasets
!pip install -q huggingface_hub

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
huggingface_token = user_secrets.get_secret("hf_token")

# Set the token as an environment variable for Hugging Face libraries to use automatically
import os
os.environ["HF_TOKEN"] = huggingface_token

# Verify the login
!huggingface-cli whoami


In [ ]:
# ライブラリのインポート
import os
import json
import torch
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from huggingface_hub import login, HfApi
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')
    print(f'CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# データファイルのパスを指定
data_path = '//kaggle/input/privatedata/terminal_care_counseling_raw_10000.jsonl'

# 方法2: 直接アップロードする場合
# from google.colab import files
# uploaded = files.upload()
# data_path = list(uploaded.keys())[0]

In [ ]:
# JSONLファイルを読み込む
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

# データを読み込み
raw_data = load_jsonl(data_path)
print(f'読み込んだデータ数: {len(raw_data)}件')

# サンプルを表示
print('\nサンプルデータ:')
print(json.dumps(raw_data[0], ensure_ascii=False, indent=2))

In [ ]:
# Hugging Face Dataset形式に変換
def prepare_dataset(data):
    formatted_data = []
    for item in data:
        formatted_data.append({
            'id': item['id'],
            'category': item['category'],
            'messages': item['messages']
        })
    return Dataset.from_list(formatted_data)

dataset = prepare_dataset(raw_data)
print(f'Dataset size: {len(dataset)}')
print(f'Features: {dataset.features}')

In [ ]:
# トレーニングセットと検証セットに分割
train_test_split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

print(f'Training set: {len(train_dataset)} samples')
print(f'Validation set: {len(eval_dataset)} samples')

In [ ]:
    # モデルID
model_id = 'google/medgemma-1.5-4b-it'

# 4-bit量子化設定（メモリ効率化）
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print('モデルを読み込んでいます...')
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
processor.tokenizer.padding_side = 'right'

print('モデルの読み込みが完了しました！')
print(f'Model device map: {model.hf_device_map}')

## LoRA設定

In [ ]:
# LoRA設定
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj',
        'o_proj',
        'k_proj',
        'v_proj',
        'gate_proj',
        'up_proj',
        'down_proj',
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

# LoRAモデルの作成
model = get_peft_model(model, peft_config)

# 学習可能なパラメータ数を表示
model.print_trainable_parameters()

## データコレーターの定義

In [ ]:
def collate_fn(examples):
    prompts = []
    
    for ex in examples:
        prompt = processor.apply_chat_template(
            ex['messages'],
            tokenize=False,
            add_generation_prompt=False
        ).strip()
        prompts.append(prompt)
    
    batch = processor(
        text=prompts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=2048
    )
    
    labels = batch['input_ids'].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch['labels'] = labels
    
    return batch

## トレーニング設定

In [ ]:
# 出力ディレクトリ
output_dir = '/kaggle/working/'

# トレーニング設定
training_args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim='adamw_torch_fused',
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    logging_steps=10,
    save_strategy='epoch',
    # evaluation_strategy='epoch',
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    remove_unused_columns=False,
    dataset_kwargs={'skip_prepare_dataset': True},
    report_to='none',
)

print('トレーニング設定:')
print(f'  Epochs: {training_args.num_train_epochs}')
print(f'  Batch size: {training_args.per_device_train_batch_size}')
print(f'  Gradient accumulation: {training_args.gradient_accumulation_steps}')
print(f'  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print(f'  Learning rate: {training_args.learning_rate}')

In [ ]:
# SFTTrainerの作成
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
    args=training_args,
)

print('トレーニングを開始します...')
print('=' * 50)

In [ ]:
# モデルを保存
final_model_dir = 'medgemma_output'

# LoRAアダプターを保存
trainer.save_model(final_model_dir)
processor.save_pretrained(final_model_dir)

print(f'モデルを保存しました: {final_model_dir}')

# Google Driveにコピー（オプション）
import shutil
drive_save_path = '/kaggle/working/'
shutil.copytree(final_model_dir, drive_save_path, dirs_exist_ok=True)
print(f'モデルを保存しました: {drive_save_path}')

In [ ]:
# テスト用のプロンプト
test_messages = [
    {
        'role': 'user',
        'content': [{'type': 'text', 'text': '最近、仕事に行くのが辛くて...毎日患者さんの最期を看取っていて、心が持ちません'}]
    }
]

# チャットテンプレートを適用
prompt = processor.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True
)

# 入力をトークナイズ
inputs = processor(
    text=prompt,
    return_tensors='pt'
).to(model.device)

# 生成
with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

# デコード
response = processor.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

print('入力:')
print(test_messages[0]['content'][0]['text'])
print('\n出力:')
print(response)

In [ ]:
# さらにテスト
test_cases = [
    '若い患者さんを看取ると、自分の子供のことを考えてしまって仕方ありません',
    '患者さんが『もう治療したくない』って言っているのに、家族が延命を希望していて...',
    '同僚との関係がギクシャクしていて、職場に行きづらいです',
    '3交代制で、生活リズムが完全に乱れてしまっています',
    '看護師としての自分に自信がなくて、毎日不安です',
]

for i, test_input in enumerate(test_cases, 1):
    print(f'\n{'='*60}')
    print(f'【テスト {i}】')
    print(f'入力: {test_input}')
    
    messages = [{'role': 'user', 'content': [{'type': 'text', 'text': test_input}]}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=prompt, return_tensors='pt').to(model.device)
    
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    
    response = processor.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    print(f'\n出力:')
    print(response[:300] + '...' if len(response) > 300 else response)